# DPO — Direct Preference Optimization (Qwen3-4B-Instruct)

Bu notebook **smoke test ile kanıtlanmış** çalışır (`smoke/05_dpo_qwen3_smoke.py`).

## Niye sadece Qwen3-4B-Instruct?

Test ettim — Unsloth 2026.4.8 DPOTrainer **multimodal modelleri text-only DPO için desteklemiyor**:

| Model | Class | DPO Sonuç |
|---|---|---|
| Qwen3.5-2B | FastVisionModel | ❌ `KeyError: 'images'` |
| Gemma 4 E2B | FastModel (multimodal) | ❌ `KeyError: 'images'` |
| **Qwen3-4B-Instruct** | **FastLanguageModel** | ✅ **Çalışıyor** |
| Qwen3-4B-Thinking | FastLanguageModel | ✅ |
| Qwen3-1.7B | FastLanguageModel | ✅ |

**Sebep:** Multimodal tokenizer DPOTrainer'in vision branch'ini tetikliyor, dataset'te `images` kolonu olmayınca patlıyor. Boş list eklemek de işe yaramıyor (`IndexError`).

Vision DPO yapmak için TRL'in `DPOTrainer` direkt kullanılabilir (Unsloth dışında) ama scope dışı.

## Donanım Doğrulaması (RTX 4070 Ti SUPER 16GB)

| Metrik | Değer |
|---|---|
| Model load | 4-bit + LoRA r=64 |
| Peak VRAM | **7.84 GB** (47% of 16.6) |
| 3-step süre | 5.3 sec |
| 200 step tahmini | ~6-7 dk |

## İçindekiler
1. **DPO Nedir?** — Tetri
2. **Setup + `PatchDPOTrainer()`**
3. **Model — Qwen3-4B-Instruct + LoRA r=64**
4. **Dataset — UltraFeedback Binarized**
5. **DPOConfig — Kritik Parametreler**
6. **DPOTrainer + Reference Model Sihri**
7. **Training**
8. **Save**
9. **Yaygın Tuzaklar**

## 1. DPO Nedir?

**Direct Preference Optimization** — RLHF'in basitleştirilmiş versiyonu. Reward model eğitmek yerine doğrudan preference data (`chosen` vs `rejected`) ile model'i optimize eder.

### Nasıl çalışır?
1. **Veri:** Her örnek: `(prompt, chosen_response, rejected_response)`
2. **Loss:** Model'in `chosen`'a verdiği logprob'u arttır, `rejected`'a verdiği logprob'u azalt
3. **Reference:** Modelin SFT versiyonu (LoRA off) — orijinalden sapmayı KL ile sınırla
4. **Beta:** KL penalty gücü (0.1 default)

### DPO vs ORPO vs KTO
| Method | Veri | Reference Model | One-step? |
|---|---|---|---|
| **DPO** | Paired (chosen, rejected) | Var (PEFT auto) | SFT sonrası |
| **ORPO** | Paired | YOK | Tek aşama (SFT+DPO bir arada) |
| **KTO** | Unpaired (good/bad label) | Var | SFT sonrası |

DPO **en yaygın** ve en iyi anlaşılmış metot.

## 2. Setup + `PatchDPOTrainer()`

**KRİTİK:** `PatchDPOTrainer()` MUTLAKA `from trl import DPOTrainer`'dan **önce** çalışmalı. Aksi halde Unsloth'un fast kernels'ı patch edilmemiş olur.

In [ ]:
import unsloth                                # MUTLAKA EN BASTA
from unsloth import PatchDPOTrainer
PatchDPOTrainer()                            # KRITIK: trl import'undan ONCE

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 3. Model — Qwen3-4B-Instruct + LoRA r=64

DPO için **r=64** önerilir (SFT'deki 16/32'den yüksek). Sebep: preference learning daha hassas adapter capacity gerektirir.

**Padding side ZORUNLU = `left`** — DPOTrainer right padding ile çalışmaz.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length = 4096,
    load_in_4bit = True,                     # 4-bit QLoRA (4GB peak)
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,                                   # DPO icin yuksek (SFT 16-32 vs DPO 64)
    lora_alpha = 64,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

# Padding side — DPOTrainer requires LEFT
tokenizer.padding_side = 'left'
print(f'Padding: {tokenizer.padding_side}')

## 4. Dataset — UltraFeedback Binarized

TRL'nin resmi DPO test dataset'i. 60k+ preference pair, conversational format. Smoke için 200 satır.

### Format (TRL 0.24 conversational)
```python
{
  'chosen':   [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': 'iyi cevap'}],
  'rejected': [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': 'kotu cevap'}],
  'score_chosen':   8.0,
  'score_rejected': 2.0,
}
```

**Önemli:** TRL 0.24 conversational format'ı **otomatik chat template uygular**. Manuel format'lamaya gerek yok!

In [ ]:
dataset = load_dataset('trl-lib/ultrafeedback_binarized', split='train[:5000]')
print(f'Dataset rows: {len(dataset)}')
print(f'Cols: {dataset.column_names}\n')

# Sample
sample = dataset[0]
print('--- Sample[0] ---')
for k, v in sample.items():
    s = str(v)[:300]
    print(f'{k}: {s}{"..." if len(str(v))>300 else ""}')

## 5. DPOConfig — Kritik Parametreler

TRL 0.24 değişiklikleri:
- `beta`, `max_length`, `max_prompt_length` artık **DPOConfig içinde** (DPOTrainer kwargs değil)
- `tokenizer=` → **`processing_class=`**

### Ana Parametreler
| Parametre | Değer | Açıklama |
|---|---|---|
| `beta` | **0.1** | KL penalty. Düşük = SFT'ye yakın kal, yüksek = preference güçlü öğren |
| `loss_type` | `sigmoid` | Default. Diğer: hinge, ipo, kto_pair, robust |
| `learning_rate` | **5e-6** | SFT'deki 2e-4'ten **40x DAHA DÜŞÜK** |
| `max_length` | 1024 | prompt + completion toplam |
| `max_prompt_length` | 512 | Yarısı prompt |
| `bf16` | True | Default DPOConfig'te |
| `optim` | `adamw_8bit` | bitsandbytes 8-bit Adam, ~2GB tasarruf |

In [ ]:
training_args = DPOConfig(
    # ----- DPO-specific -----
    beta = 0.1,
    loss_type = 'sigmoid',
    max_length = 1024,
    max_prompt_length = 512,
    # ----- HF Trainer -----
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,         # effective batch = 8
    learning_rate = 5e-6,                    # DPO icin DUSUK
    warmup_ratio = 0.1,
    num_train_epochs = 1,                    # DPO 1-3 epoch arasi yeterli
    logging_steps = 10,
    optim = 'adamw_8bit',
    weight_decay = 0.0,
    lr_scheduler_type = 'linear',
    bf16 = True,
    seed = 3407,
    output_dir = 'outputs_dpo',
    report_to = 'none',
)

## 6. DPOTrainer + Reference Model Sihri

**`ref_model=None`** + PEFT model = **otomatik LoRA toggle** ile referans logprob hesaplar. İkinci model yüklenmez!

Detay:
- Forward pass 1: LoRA aktif → policy logprobs
- Forward pass 2: LoRA disabled → reference logprobs (base model)
- Memory tasarruf: 2x model değil, sadece +1-2 GB activation overhead

### `train_on_responses_only` KULLANMA!
DPO zaten prompt'u maskeliyor (loss sadece completion üzerinde). Eklemek hata verir.

In [ ]:
trainer = DPOTrainer(
    model = model,
    ref_model = None,                        # PEFT auto-handles (KEY!)
    args = training_args,
    train_dataset = dataset,
    processing_class = tokenizer,            # NOT 'tokenizer=' (TRL 0.24 rename)
)

## 7. Training

İzlenecek metrikler:
- **`loss`** — başlangıç ~0.6931 (=ln(2), random), düşmesi gerek
- **`rewards/chosen`** — pozitif olmalı (model chosen'i tercih ediyor)
- **`rewards/rejected`** — negatif olmalı (model rejected'i istemiyor)
- **`rewards/margins`** — chosen - rejected, **pozitif olmalı** (öğreniyor)
- **`rewards/accuracies`** — chosen > rejected oranı, 0.5'ten yukarı çıkmalı

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name} | start mem = {start_mem} / {max_mem} GB')

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n{trainer_stats.metrics['train_runtime']:.2f} sec")
print(f"Train loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f'Peak VRAM: {used} GB')

for k in ['train_rewards/chosen', 'train_rewards/rejected', 'train_rewards/margins', 'train_rewards/accuracies']:
    if k in trainer_stats.metrics:
        print(f'{k}: {trainer_stats.metrics[k]:.4f}')

## 8. Save

In [ ]:
# A. LoRA adapter (en kucuk)
model.save_pretrained('qwen3_dpo_lora')
tokenizer.save_pretrained('qwen3_dpo_lora')
print('LoRA saved: qwen3_dpo_lora/')

# B. Merged 16-bit (vLLM/HF)
# model.save_pretrained_merged(
#     'qwen3_dpo_merged', tokenizer, save_method='merged_16bit',
# )

# C. GGUF
# model.save_pretrained_gguf('qwen3_dpo_gguf', tokenizer, quantization_method='q4_k_m')

## 9. Yaygın Tuzaklar

| # | Hata | Sonuç | Çözüm |
|---|---|---|---|
| 1 | **Multimodal model + DPO** | `KeyError: 'images'` veya `IndexError` | Sadece text-only model (FastLanguageModel) |
| 2 | `PatchDPOTrainer()` çağırmamak | Yavaş training (Unsloth fast kernels yok) | Önce çağır, sonra `from trl import DPOTrainer` |
| 3 | `tokenizer=tok` (TRL 0.22 API) | `TypeError: unexpected kwarg 'tokenizer'` | `processing_class=tok` (TRL 0.24) |
| 4 | `beta` DPOTrainer kwarg'ı | TypeError | DPOConfig içinde geçir |
| 5 | `padding_side='right'` | DPO loss yanlış hesaplanır | `tokenizer.padding_side = 'left'` |
| 6 | `train_on_responses_only` ekle | DPO zaten prompt'u maskeliyor, hata | DPO için kullanma |
| 7 | `learning_rate=2e-4` (SFT default) | Reward hacking, model collapse | DPO için 5e-6 / 1e-5 |
| 8 | `force_use_ref_model=True` | 2x VRAM (separate ref loaded) | Default `False` bırak |
| 9 | `sync_ref_model=True` + PEFT | Çakışır (PEFT'te ref ayrı yok) | Default `False` |
| 10 | `beta` çok yüksek (>0.5) | Model çok az değişir (KL cezası ağır) | 0.1-0.2 başla |
| 11 | `beta` çok düşük (<0.01) | Model SFT'den çok uzaklaşır, collapse | 0.05+ tut |

## Özet

1. **DPO = SFT + preference optimization** — chosen'a +, rejected'a - (KL ile sınırlı)
2. **`PatchDPOTrainer()` ZORUNLU**, TRL import'undan önce
3. **Multimodal modelleri kullanma** — Unsloth DPO sadece text-only çalışıyor şu anda
4. **Padding LEFT, lr=5e-6, beta=0.1, r=64** — kanonik DPO recipe
5. **`ref_model=None` + PEFT** = otomatik LoRA toggle (2x model yüklenmez)
6. **VRAM 7.84 GB** (Qwen3-4B 4-bit + LoRA r=64) — 16GB'da çok rahat

**Test edildi:** `smoke/05_dpo_qwen3_smoke.py` — pipeline 3 step ile doğrulandı (loss=0.6931 → 0.745, ki bu fluctuation normal 3 step için).

## Sonraki Adımlar
- **ORPO:** SFT olmadan tek aşamada — reference model gerekmez
- **GRPO:** RL approach — reasoning için ideal
- **KTO:** Unpaired binary feedback (good/bad)